In [10]:
# =============================================================================
# Fig2: Premature deaths & mortality rate by health-service (HS) group
# Layout: 2 rows × 3 cols
#   Row 0: mort_2020 | mort_2060 | delta_mort
#   Row 1: mortratio_2020 | mortratio_2060 | delta_mortratio
#
# Each panel:
#   • Density curves (top, independent area, stacked per HS group)
#   • Jittered scatter (horizontal, per group)
#   • Horizontal boxplot with T-shaped whisker caps
# =============================================================================

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec

# ── 1. 参数配置 ───────────────────────────────────────────────────────────────

XLSX_PATH = Path("/Volumes/UCL/论文工作/空气污染/清华城市健康指数-健康服务.xlsx")
OUTFILE   = Path("/Users/shirley/Desktop/plots_V2/Fig2.png")
SHEET     = "earlypeak_NZ_CL"

TOTAL_W_MM, TOTAL_H_MM, DPI = 180, 120, 300   # 与 Fig1 一致
W_IN = 18 / 2.54   # 加宽：180mm → 220mm
H_IN = 10 / 2.54

# HS 分组顺序与颜色 — 单色系顺序渐变（ColorBrewer RdYlBu 5级，体现 Poor→Excellent 梯度）
HS_LEVELS = ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]
COL_GROUP = {
    "Poor":         "#d7191c",
    "Moderate":     "#fdae61",
    "Intermediate": "#ffffbf",
    "Good":         "#abd9e9",
    "Excellent":    "#2c7bb6",
}

# Y 轴位置：levels / 3，与 R 版 y_pos = as.numeric(HS_type) / 3 对应
Y_POS = {lvl: (i + 1) / 3 for i, lvl in enumerate(HS_LEVELS)}
UNIQUE_Y = sorted(Y_POS.values())

# 密度图区域压缩（原 +2.2 → +1.2），减少密度曲线占比
DENSITY_MIN = max(UNIQUE_Y) + 0.2
DENSITY_MAX = DENSITY_MIN + 0.8

# ── 2. 读取数据 ───────────────────────────────────────────────────────────────

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET, usecols=[
    "city", "mort_2020", "mort_2060", "delta_mort",
    "mortratio_2020", "mortratio_2060", "delta_mortratio",
    "city_size", "GDP_group", "HS", "region_HW", "HS_type2",
])

df["HS_type"] = pd.Categorical(df["HS_type2"], categories=HS_LEVELS, ordered=True)
df = df.dropna(subset=["HS_type"])
df["y_pos"] = df["HS_type"].map(Y_POS)

# ── 3. 全局样式 ───────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.titlesize":   6,
    "axes.titleweight": "bold",
    "axes.titlepad":    4,
    "axes.labelsize":   6,
    "xtick.labelsize":  5.5,
    "ytick.labelsize":  5.5,
})

# ── 4. 单面板绘制函数 ─────────────────────────────────────────────────────────

def make_panel(ax, data, xvar, xlabel, xlim, show_y_labels=False):
    """
    在 ax 上绘制一个 HS 分组的横向分布面板：
      • 顶部：各组 KDE 密度曲线（填色 + 线）
      • 中部：jitter 散点
      • 中部：横向箱线图 + T 形须帽
    """
    rng = np.random.default_rng(42)

    for lvl in HS_LEVELS:
        grp    = data[data["HS_type"] == lvl][xvar].dropna()
        grp    = grp[np.isfinite(grp)]
        y_pos  = Y_POS[lvl]
        color  = COL_GROUP[lvl]

        if len(grp) < 2:
            continue

        # ── 小提琴（在最底层，zorder=1） ──
        vp = ax.violinplot(grp, positions=[y_pos], vert=False,
                           widths=0.28, showextrema=False)
        for body in vp["bodies"]:
            body.set_facecolor(color)
            body.set_alpha(0.15)
            body.set_edgecolor("none")

        # ── Jitter 散点 ──
        jitter = rng.uniform(-0.06, 0.06, size=len(grp))
        ax.scatter(grp, y_pos + jitter,
                   color=color, s=1.2, alpha=0.20, linewidths=0, zorder=2)

        # ── 箱线统计量 ──
        q1, med, q3 = np.percentile(grp, [25, 50, 75])
        iqr  = q3 - q1
        lo   = grp[grp >= q1 - 1.5 * iqr].min()   # 下须（最小非离群值）
        hi   = grp[grp <= q3 + 1.5 * iqr].max()   # 上须

        box_h   = 0.06                   # 箱体半高（缩小）
        cap_h   = 0.03                   # T 形须帽半高

        # 箱体
        rect = mpatches.FancyBboxPatch(
            (q1, y_pos - box_h), q3 - q1, 2 * box_h,
            boxstyle="square,pad=0",
            facecolor="white", edgecolor=color,
            linewidth=0.8, alpha=0.9, zorder=3,
        )
        ax.add_patch(rect)

        # 中位线
        ax.plot([med, med], [y_pos - box_h, y_pos + box_h],
                color=color, linewidth=1.2, zorder=4)

        # 须线
        ax.plot([lo, q1], [y_pos, y_pos], color=color, linewidth=0.8, zorder=3)
        ax.plot([q3, hi], [y_pos, y_pos], color=color, linewidth=0.8, zorder=3)

        # T 形须帽
        for cap_x in [lo, hi]:
            ax.plot([cap_x, cap_x], [y_pos - cap_h, y_pos + cap_h],
                    color=color, linewidth=0.8, zorder=3)

    # ── 坐标轴设置 ──
    ax.set_xlim(xlim)
    ax.set_ylim(min(UNIQUE_Y) - 0.2, max(UNIQUE_Y) + 0.2)   # 仅散点+箱线范围，去掉密度区域
    ax.set_yticks(UNIQUE_Y)
    ax.set_yticklabels(HS_LEVELS if show_y_labels else [""] * len(UNIQUE_Y))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(None)

    # 格式化 x 轴：mort 系列（第一行）→ 整数 K；mortratio 系列（第二行）→ 两位小数
    if "ratio" in xvar:
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x:.2f}")
        )
    else:
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k")
        )

    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.3)
    ax.spines["left"].set_linewidth(0.3)
    ax.grid(False)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)  # length=0 去掉短线

# ── 5. 洛伦兹曲线 + 基尼系数函数 ────────────────────────────────────────────

def lorenz_curve(values):
    """返回 (cum_share_population, cum_share_values)，均从 0 到 1。"""
    v = np.array(values, dtype=float)
    v = v[np.isfinite(v) & (v >= 0)]
    v = np.sort(v)
    cum_v = np.cumsum(v)
    n = len(v)
    x = np.linspace(0, 1, n + 1)
    y = np.concatenate([[0], cum_v / cum_v[-1]])
    return x, y


def gini(values):
    """基尼系数 = 1 - 2 * AUC(Lorenz curve)。"""
    x, y = lorenz_curve(values)
    auc = np.trapz(y, x)
    return 1 - 2 * auc


def draw_lorenz_panel(ax, col_2020, col_2060, y_label, tag):
    """
    洛伦兹曲线子图：
      • 对角线（完全平等参考线）
      • 两条洛伦兹曲线：2020（橙）和 2060（蓝）
      • 各自标注基尼系数
    """
    COL_20 = "#fdb462"    # 与 Fig1 violin 2020 一致
    COL_60 = "#80b1d3"    # 与 Fig1 violin 2060 一致

    # 对角线
    ax.plot([0, 1], [0, 1], color="grey", linewidth=0.8,
            linestyle="--", zorder=1)

    for col, color, year in [
        (col_2020, COL_20, "2020"),
        (col_2060, COL_60, "2060"),
    ]:
        v  = df[col].dropna().values
        x, y = lorenz_curve(v)
        g  = gini(v)

        # 填充曲线与对角线之间的面积
        ax.fill_between(x, x, y, color=color, alpha=0.08)
        ax.plot(x, y, color=color, linewidth=1.2,
                label=f"{year}  Gini={g:.3f}", zorder=3)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal", adjustable="box")   # 强制 xy 等长

    ax.set_xlabel("Cumulative share of cities", fontsize=6, labelpad=2)
    # y 轴标签用 ax.text 固定在轴左侧，避免 adjustable="box" 后被推远
    ax.set_ylabel("")
    ax.text(-0.18, 0.5, y_label,
            transform=ax.transAxes, fontsize=6,
            rotation=90, va="center", ha="center")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(0.3)
    ax.grid(linewidth=0.2, color="#e0e0e0", zorder=0)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)

    ax.legend(fontsize=5, frameon=False, loc="upper left",
              handlelength=1.2, handletextpad=0.4, labelspacing=0.3)

    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="bottom", ha="left")


# ── 6. 图形拼合（2行 × 4列） ─────────────────────────────────────────────────

fig = plt.figure(figsize=(W_IN, H_IN), dpi=DPI)

gs = GridSpec(
    nrows=2, ncols=4,
    figure=fig,
    width_ratios=[0.8, 0.8, 0.8, 1],   # 前三列分布图，第四列洛伦兹曲线
    hspace=0.35,
    wspace=0.22,
)

axes = [[fig.add_subplot(gs[r, c]) for c in range(4)] for r in range(2)]

# 面板定义：(xvar, xlabel, xlim, show_y_labels)
PANELS = [
    [
        ("mort_2020",      "Premature deaths in 2020",  (0, 35000),      True),
        ("mort_2060",      "Premature deaths in 2060",  (0, 35000),      False),
        ("delta_mort",     "Change in deaths",          (-20000, 20000), False),
    ],
    [
        ("mortratio_2020",   "Mortality rate in 2020 (%)",  (0, 0.1),    True),
        ("mortratio_2060",   "Mortality rate in 2060 (%)",  (0, 0.1),    False),
        ("delta_mortratio",  "Change in rate (%)",          (-0.05, 0.05), False),
    ],
]

TAGS = [["a", "b", "c", "d"], ["e", "f", "g", "h"]]

for r in range(2):
    for c in range(3):
        xvar, xlabel, xlim, show_y = PANELS[r][c]
        ax = axes[r][c]
        make_panel(ax, df, xvar, xlabel, xlim, show_y_labels=show_y)
        ax.text(-0.14 if show_y else -0.05, 1.03, TAGS[r][c],
                transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="bottom", ha="left")

# 第四列：洛伦兹曲线
draw_lorenz_panel(axes[0][3], "mort_2020",      "mort_2060",
                  "Cumulative share of deaths",        "d")
draw_lorenz_panel(axes[1][3], "mortratio_2020", "mortratio_2060",
                  "Cumulative share of mortality rate", "h")

# ── 高度对齐：让 a/b/c 和 e/f/g 的高度与 d/h 一致，d/h 保持正方形 ──────────
#
# set_aspect("equal") 生效后，d/h 的实际高度由其宽度决定，
# 需要 fig.canvas.draw() 触发布局计算，才能读到正确的 Bbox。
fig.canvas.draw()

def _match_height_to_ref(row_axes, ref_ax):
    """
    将 row_axes（list）的高度和垂直中心对齐到 ref_ax。
    保持各自的 x0 和 width 不变，只调整 y0 和 height。
    """
    ref_pos = ref_ax.get_position()
    ref_cy  = ref_pos.y0 + ref_pos.height / 2
    ref_h   = ref_pos.height

    for ax in row_axes:
        pos = ax.get_position()
        new_y0 = ref_cy - ref_h / 2
        ax.set_position([pos.x0, new_y0, pos.width, ref_h])

_match_height_to_ref([axes[0][0], axes[0][1], axes[0][2]], axes[0][3])
_match_height_to_ref([axes[1][0], axes[1][1], axes[1][2]], axes[1][3])

# ── 7. HS 分组图例（底部，1行） ───────────────────────────────────────────────

legend_handles = [
    mpatches.Patch(facecolor=COL_GROUP[lvl], edgecolor=COL_GROUP[lvl],
                   alpha=0.7, label=lvl)
    for lvl in HS_LEVELS
]
# fig.legend(
#     handles=legend_handles,
#     title=None,
#     fontsize=6,
#     loc="lower center",
#     bbox_to_anchor=(0.5, -0.04),
#     frameon=False,
#     ncol=len(HS_LEVELS),
#     handlelength=1.0,
#     columnspacing=1.0,
# )

# ── 7. 保存 ───────────────────────────────────────────────────────────────────

OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"✓ Saved → {OUTFILE}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_17396/107431291.py:180: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(y, x)


✓ Saved → /Users/shirley/Desktop/plots_V2/Fig2.png


In [11]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec

# ── 1. 路径配置 ───────────────────────────────────────────────────────────────

XLSX_PATH = Path("/Volumes/UCL/论文工作/空气污染/清华城市健康指数-健康服务.xlsx")
OUTFILE   = Path("/Users/shirley/Desktop/plots_V2/Fig2_spline.png")
SHEET     = "earlypeak_NZ_CL"

GAM_PRED  = Path("/Users/shirley/Desktop/plots_V2/gam_pred.csv")
GAM_RUG   = Path("/Users/shirley/Desktop/plots_V2/gam_rug.csv")

W_IN = 18 / 2.54
H_IN = 15 / 2.54          # 加高以容纳第三行
DPI  = 300

# ── 2. HS 分组配置 ────────────────────────────────────────────────────────────

HS_LEVELS = ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]
COL_GROUP = {
    "Poor":         "#d7191c",
    "Moderate":     "#f47920",
    "Intermediate": "#f6c101",
    "Good":         "#abd9e9",
    "Excellent":    "#2c7bb6",
}
Y_POS    = {lvl: (i + 1) / 3 for i, lvl in enumerate(HS_LEVELS)}
UNIQUE_Y = sorted(Y_POS.values())

# ── 3. 读取数据 ───────────────────────────────────────────────────────────────

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET, usecols=[
    "city", "mort_2020", "mort_2060", "delta_mort",
    "mortratio_2020", "mortratio_2060", "delta_mortratio",
    "city_size", "GDP_group", "HS", "region_HW", "HS_type2",
])
df["HS_type"] = pd.Categorical(df["HS_type2"], categories=HS_LEVELS, ordered=True)
df = df.dropna(subset=["HS_type"])
df["y_pos"] = df["HS_type"].map(Y_POS)

gam_pred = pd.read_csv(GAM_PRED)
gam_rug  = pd.read_csv(GAM_RUG)

# ── 4. 全局样式 ───────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.titlesize":   6,
    "axes.titleweight": "bold",
    "axes.titlepad":    4,
    "axes.labelsize":   6,
    "xtick.labelsize":  5.5,
    "ytick.labelsize":  5.5,
})

# ── 5. 面板函数：分布图 ───────────────────────────────────────────────────────

def make_panel(ax, data, xvar, xlabel, xlim, show_y_labels=False):
    rng = np.random.default_rng(42)
    for lvl in HS_LEVELS:
        grp   = data[data["HS_type"] == lvl][xvar].dropna()
        grp   = grp[np.isfinite(grp)]
        y_pos = Y_POS[lvl]
        color = COL_GROUP[lvl]
        if len(grp) < 2:
            continue
        vp = ax.violinplot(grp, positions=[y_pos], vert=False,
                           widths=0.28, showextrema=False)
        for body in vp["bodies"]:
            body.set_facecolor(color); body.set_alpha(0.15); body.set_edgecolor("none")
        jitter = rng.uniform(-0.06, 0.06, size=len(grp))
        ax.scatter(grp, y_pos + jitter, color=color, s=1.2,
                   alpha=0.20, linewidths=0, zorder=2)
        q1, med, q3 = np.percentile(grp, [25, 50, 75])
        iqr = q3 - q1
        lo  = grp[grp >= q1 - 1.5*iqr].min()
        hi  = grp[grp <= q3 + 1.5*iqr].max()
        box_h, cap_h = 0.06, 0.03
        rect = mpatches.FancyBboxPatch(
            (q1, y_pos - box_h), q3 - q1, 2*box_h,
            boxstyle="square,pad=0",
            facecolor="white", edgecolor=color,
            linewidth=0.8, alpha=0.9, zorder=3)
        ax.add_patch(rect)
        ax.plot([med, med], [y_pos-box_h, y_pos+box_h],
                color=color, linewidth=1.2, zorder=4)
        ax.plot([lo, q1], [y_pos, y_pos], color=color, linewidth=0.8, zorder=3)
        ax.plot([q3, hi], [y_pos, y_pos], color=color, linewidth=0.8, zorder=3)
        for cap_x in [lo, hi]:
            ax.plot([cap_x, cap_x], [y_pos-cap_h, y_pos+cap_h],
                    color=color, linewidth=0.8, zorder=3)
    ax.set_xlim(xlim)
    ax.set_ylim(min(UNIQUE_Y)-0.2, max(UNIQUE_Y)+0.2)
    ax.set_yticks(UNIQUE_Y)
    ax.set_yticklabels(HS_LEVELS if show_y_labels else [""]*len(UNIQUE_Y))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(None)
    if "ratio" in xvar:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.2f}"))
    else:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.3)
    ax.spines["left"].set_linewidth(0.3)
    ax.grid(False)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)

# ── 6. 面板函数：洛伦兹曲线 ──────────────────────────────────────────────────

def lorenz_curve(values):
    v = np.array(values, dtype=float)
    v = v[np.isfinite(v) & (v >= 0)]
    v = np.sort(v)
    cum_v = np.cumsum(v)
    x = np.linspace(0, 1, len(v)+1)
    y = np.concatenate([[0], cum_v / cum_v[-1]])
    return x, y

def gini(values):
    x, y = lorenz_curve(values)
    return 1 - 2*np.trapz(y, x)

def draw_lorenz_panel(ax, col_2020, col_2060, y_label, tag):
    COL_20, COL_60 = "#fdb462", "#80b1d3"
    ax.plot([0,1],[0,1], color="grey", linewidth=0.8, linestyle="--", zorder=1)
    for col, color, year in [(col_2020, COL_20, "2020"), (col_2060, COL_60, "2060")]:
        v = df[col].dropna().values
        x, y = lorenz_curve(v)
        g = gini(v)
        ax.fill_between(x, x, y, color=color, alpha=0.08)
        ax.plot(x, y, color=color, linewidth=1.2, label=f"{year}  Gini={g:.3f}", zorder=3)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Cumulative share of cities", fontsize=6, labelpad=2)
    ax.set_ylabel("")
    ax.text(-0.18, 0.5, y_label, transform=ax.transAxes, fontsize=6,
            rotation=90, va="center", ha="center")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_linewidth(0.3)
    ax.grid(linewidth=0.2, color="#e0e0e0", zorder=0)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)
    ax.legend(fontsize=5, frameon=False, loc="upper left",
              handlelength=1.2, handletextpad=0.4, labelspacing=0.3)
    ax.text(-0.12, 1.03, tag, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="bottom", ha="left")

# ── 7. 面板函数：GAM 样条 ─────────────────────────────────────────────────────

COLOR_FULL = "#d6604d"
COLOR_SUB  = "#4db3b3"

def fmt_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

def draw_spline_panel(ax, gam_pred, gam_rug, yvar, ylabel, tag, show_y_label=True):
    pf = gam_pred[(gam_pred["yvar"] == yvar) & (gam_pred["sample"] == "full")]
    ps = gam_pred[(gam_pred["yvar"] == yvar) & (gam_pred["sample"] == "sub")]

    edf_full = pf["edf"].iloc[0];   p_full = pf["p_val"].iloc[0]
    edf_sub  = ps["edf"].iloc[0];   p_sub  = ps["p_val"].iloc[0]

    ax.axhline(0, linestyle="--", color="grey", linewidth=0.8, zorder=1)

    ax.fill_between(pf["HS"], pf["lo"], pf["hi"],
                    color=COLOR_FULL, alpha=0.20, linewidth=0)
    ax.fill_between(ps["HS"], ps["lo"], ps["hi"],
                    color=COLOR_SUB,  alpha=0.20, linewidth=0)

    ax.plot(pf["HS"], pf["fit"], color=COLOR_FULL, linewidth=1.0, label="All cities")
    ax.plot(ps["HS"], ps["fit"], color=COLOR_SUB,  linewidth=1.0, label="HS ≤ 90")

    # rug
    ax.autoscale(False)
    ymin, ymax = ax.get_ylim()
    rug_y = ymin - 0.02*(ymax - ymin)
    ax.plot(gam_rug["HS"].values, np.full(len(gam_rug), rug_y),
            linestyle="none", marker="|", color="black",
            alpha=0.6, markersize=3, markeredgewidth=0.4, clip_on=False)

    # annotation
    all_hi = np.concatenate([pf["hi"].values, ps["hi"].values])
    x_anno = gam_pred["HS"].min() + 0.02*(gam_pred["HS"].max() - gam_pred["HS"].min())
    y_anno = np.nanmax(all_hi) * 0.95
    ax.text(x_anno, y_anno,
            f"All cities: edf = {edf_full}, {fmt_p(p_full)}\n"
            f"HS ≤ 90:      edf = {edf_sub}, {fmt_p(p_sub)}",
            ha="left", va="top", fontsize=5, fontfamily="Arial")

    ax.set_xlabel("Health capacity (HS)", fontsize=6)
    ax.set_ylabel(ylabel if show_y_label else "", fontsize=6)
    ax.tick_params(axis="both", which="both", labelsize=5, length=0)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.3)
    ax.spines["left"].set_linewidth(0.3)

    ax.text(-0.14 if show_y_label else -0.05, 1.03, tag,
            transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="bottom", ha="left")

# ── 8. 图形拼合（3行 × 4列） ─────────────────────────────────────────────────

fig = plt.figure(figsize=(W_IN, H_IN), dpi=DPI)

gs = GridSpec(
    nrows=3, ncols=4,
    figure=fig,
    width_ratios=[0.8, 0.8, 0.8, 1],
    height_ratios=[1, 1, 1],          # 三行等高
    hspace=0.45,
    wspace=0.22,
)

# 前两行：原 Fig2
axes = [[fig.add_subplot(gs[r, c]) for c in range(4)] for r in range(2)]

PANELS = [
    [
        ("mort_2020",    "Premature deaths in 2020", (0, 35000),       True),
        ("mort_2060",    "Premature deaths in 2060", (0, 35000),        False),
        ("delta_mort",   "Change in deaths",         (-20000, 20000),   False),
    ],
    [
        ("mortratio_2020",  "Mortality rate in 2020 (%)", (0, 0.1),     True),
        ("mortratio_2060",  "Mortality rate in 2060 (%)", (0, 0.1),     False),
        ("delta_mortratio", "Change in rate (%)",         (-0.05, 0.05), False),
    ],
]
TAGS = [["a","b","c","d"], ["e","f","g","h"]]

for r in range(2):
    for c in range(3):
        xvar, xlabel, xlim, show_y = PANELS[r][c]
        ax = axes[r][c]
        make_panel(ax, df, xvar, xlabel, xlim, show_y_labels=show_y)
        ax.text(-0.14 if show_y else -0.05, 1.03, TAGS[r][c],
                transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="bottom", ha="left")

draw_lorenz_panel(axes[0][3], "mort_2020",      "mort_2060",
                  "Cumulative share of deaths",        "d")
draw_lorenz_panel(axes[1][3], "mortratio_2020", "mortratio_2060",
                  "Cumulative share of mortality rate", "h")

# 第三行：spline 面板（各跨两列）
ax_s1 = fig.add_subplot(gs[2, 0:2])   # delta_mort，占前两列
ax_s2 = fig.add_subplot(gs[2, 2:4])   # delta_mortratio，占后两列

draw_spline_panel(ax_s1, gam_pred, gam_rug,
                  yvar="delta_mort",
                  ylabel="Change in mortality",
                  tag="i", show_y_label=True)

draw_spline_panel(ax_s2, gam_pred, gam_rug,
                  yvar="delta_mortratio",
                  ylabel="Change in mortality ratio",
                  tag="j", show_y_label=True)

# spline 共享图例
handles, labels = ax_s1.get_legend_handles_labels()
fig.legend(handles, labels,
           loc="lower center", ncol=2,
           fontsize=6, frameon=False,
           bbox_to_anchor=(0.5, -0.01))

# ── 9. 高度对齐（前两行） ─────────────────────────────────────────────────────

fig.canvas.draw()

def _match_height_to_ref(row_axes, ref_ax):
    ref_pos = ref_ax.get_position()
    ref_cy  = ref_pos.y0 + ref_pos.height / 2
    ref_h   = ref_pos.height
    for ax in row_axes:
        pos = ax.get_position()
        ax.set_position([pos.x0, ref_cy - ref_h/2, pos.width, ref_h])

_match_height_to_ref([axes[0][0], axes[0][1], axes[0][2]], axes[0][3])
_match_height_to_ref([axes[1][0], axes[1][1], axes[1][2]], axes[1][3])

# ── 10. 保存 ──────────────────────────────────────────────────────────────────

OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"✓ Saved → {OUTFILE}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_17396/2464051546.py:127: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return 1 - 2*np.trapz(y, x)


✓ Saved → /Users/shirley/Desktop/plots_V2/Fig2_spline.png
